# Homework

## Module 5:  Spark and Batch Processing

<hr>

### Table of content

<hr>

- [Question 1](#question-1)
- [Question 2](#question-2)
- [Question 3](#question-3)
- [Question 4](#question-4)
- [Question 5](#question-5)
- [Question 5](#question-5)
- [Question 6](#question-6)

In [ ]:
import pyspark
from pyspark.sql import SparkSession
from pathlib import Path
from pyspark.sql import functions as F

### Question 1

Question 1: Install Spark and PySpark

    Install Spark
    Run PySpark
    Create a local spark session
    Execute spark.version.

What's the output?

After installing Java and adding pyspark to the uv environment (essentially following along the installation guide) I executed following script to find out the version of spark that I am using.

In [4]:
# Create a local spark session
spark = SparkSession.builder \
    .master("local[*]") \
    .appName("test")\
    .getOrCreate()

# Execute spark.version
print(spark.version)

4.1.1


The output is the version of pyspark I am currently using, namely <span style = "background-color: lightgreen; color: black">4.1.1</span>

<hr>

### Question 2

Read the November 2025 Yellow into a Spark Dataframe.

Repartition the Dataframe to 4 partitions and save it to parquet.

What is the average size of the Parquet (ending with .parquet extension) Files that were created (in MB)? Select the answer which most closely matches.

- <span style = "background-color: red; color white; opacity: .7">6MB</span>
- <span style = "background-color: green; color white; opacity: .7">25MB</span>
- <span style = "background-color: red; color white; opacity: .7">75MB</span>
- <span style = "background-color: red; color white; opacity: .7">100MB</span>

In [1]:
!curl -o yellow_tripdata_2025-11.parquet https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed

  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
  1 67.8M    1  960k    0     0  1374k      0  0:00:50 --:--:--  0:00:50 1393k
  8 67.8M    8 6027k    0     0  3584k      0  0:00:19  0:00:01  0:00:18 3604k
 14 67.8M   14 9814k    0     0  3667k      0  0:00:18  0:00:02  0:00:16 3680k
 21 67.8M   21 14.6M    0     0  4081k      0  0:00:17  0:00:03  0:00:14 4092k
 28 67.8M   28 19.4M    0     0  4258k      0  0:00:16  0:00:04  0:00:12 4267k
 36 67.8M   36 24.6M    0     0  4445k      0  0:00:15  0:00:05  0:00:10 4874k
 39 67.8M   39 27.0M    0     0  4070k      0  0:00:17  0:00:06  0:00:11 4230k
 45 67.8M   45 30.5M    0     0  4080k      0  0:00:17  0:00:07  0:00:10 4301k
 50 67.8M   50 34.2M    0     0  4037k      0  0:00:17  0:00:08  0:00:09 4004k
 56 67.8M   56 38.3M    0     0  4050k      0  0:00

Partitioning file in 4 partitions and finding out the size

In [2]:
from pyspark.sql import SparkSession
from pathlib import Path

try:
    spark.stop()
except:
    pass

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("q2")
    .config("spark.hadoop.mapreduce.fileoutputcommitter.algorithm.version", "2")
    .getOrCreate()
)

df = spark.read.parquet("yellow_tripdata_2025-11.parquet")
out_dir = "yellow_2025_11_repartitioned_4"

df.repartition(4).write.mode("overwrite").parquet(out_dir)

files = list(Path(out_dir).glob("*.parquet"))
sizes_mb = [f.stat().st_size / (1024 * 1024) for f in files]
print("files:", len(files))
print("avg MB:", round(sum(sizes_mb) / len(sizes_mb), 2))
print("sizes MB:", [round(x, 2) for x in sizes_mb])

files: 4
avg MB: 24.42
sizes MB: [24.4, 24.4, 24.42, 24.43]


The solution is that each partition is ~24 - 25 Mb in size

<hr>

### Question 3

How many taxi trips were there on the 15th of November?

Consider only trips that started on the 15th of November.


- <span style = "background-color: red; color white; opacity: .7">62,610</span>
- <span style = "background-color: red; color white; opacity: .7">102,340</span>
- <span style = "background-color: green; color white; opacity: .7">162,604</span>
- <span style = "background-color: red; color white; opacity: .7">225,768</span>

In [ ]:

df = spark.read.parquet("yellow_tripdata_2025-11.parquet")

trips_15_nov = (
    df.filter(F.to_date("tpep_pickup_datetime") == F.lit("2025-11-15"))
      .count()
)

print(trips_15_nov)

162604


The correct answer are 162604 trips.

<hr>

### Question 4

What is the length of the longest trip in the dataset in hours?

- <span style = "background-color: red; color white; opacity: .7">22.7</span>
- <span style = "background-color: red; color white; opacity: .7">58.2</span>
- <span style = "background-color: green; color white; opacity: .7">90.6</span>
- <span style = "background-color: red; color white; opacity: .7">134.5</span>

In [7]:
longest_trip_hours = (
    df.withColumn(
        "trip_hours",
        (
            F.to_unix_timestamp("tpep_dropoff_datetime")
            - F.to_unix_timestamp("tpep_pickup_datetime")
        ) / 3600.0
    )
    .filter(F.col("trip_hours") >= 0)  # optional: ignore bad negative records
    .agg(F.max("trip_hours").alias("max_trip_hours"))
    .first()["max_trip_hours"]
)

print(round(longest_trip_hours, 1))

90.6


The longest trip in hours was 90.6 hours (where did they go?)

<hr>

### Question 5

Spark's User Interface which shows the application's dashboard runs on which local port?


- <span style = "background-color: red; color white; opacity: .7">80</span>
- <span style = "background-color: red; color white; opacity: .7">443</span>
- <span style = "background-color: green; color white; opacity: .7">4040</span>
- <span style = "background-color: red; color white; opacity: .7">8080</span>

In the lecture videos the port 4040 was used to show the application's dashboard.

### Question 6


Using the zone lookup data and the Yellow November 2025 data, what is the name of the LEAST frequent pickup location Zone?


- <span style = "background-color: green; color white; opacity: .7">Governor's Island/Ellis Island/Liberty Island</span>
- <span style = "background-color: green; color white; opacity: .7">Arden Heights</span>
- <span style = "background-color: red; color white; opacity: .7">Rikers Island</span>
- <span style = "background-color: red; color white; opacity: .7">Jamaica Bay</span>

I had the taxi zone lookup data already installed on the repository so I just copied it in the folder

In [8]:
trips = spark.read.parquet("yellow_tripdata_2025-11.parquet")

# Taxi zone lookup
zones = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("taxi_zone_lookup.csv")
)

least_frequent_pickup = (
    trips.join(zones, trips.PULocationID == zones.LocationID, "left")
         .groupBy("Zone")
         .count()
         .orderBy(F.col("count").asc(), F.col("Zone").asc())
)

least_frequent_pickup.show(10, truncate=False)

+---------------------------------------------+-----+
|Zone                                         |count|
+---------------------------------------------+-----+
|Arden Heights                                |1    |
|Eltingville/Annadale/Prince's Bay            |1    |
|Governor's Island/Ellis Island/Liberty Island|1    |
|Port Richmond                                |3    |
|Great Kills                                  |4    |
|Green-Wood Cemetery                          |4    |
|Rikers Island                                |4    |
|Rossville/Woodrow                            |4    |
|Jamaica Bay                                  |5    |
|Westerleigh                                  |12   |
+---------------------------------------------+-----+
only showing top 10 rows


The right answer is Arden Heights, Eltingville/Annadale/Prince's Bay and Govenor's island/Ellis Island/ Liberty Island